# Quickstart: Your First Bayesian VAR

This tutorial walks you through fitting a Bayesian VAR model to macroeconomic data using Litterman.

In [1]:
import numpy as np
import pandas as pd

from litterman import VAR, VARData, select_lag_order
from litterman.samplers import NUTSSampler

## Create some synthetic data

We'll simulate a simple VAR(1) process with two variables.

In [2]:
rng = np.random.default_rng(42)
T = 200
y = np.zeros((T, 2))
for t in range(1, T):
    y[t] = 0.5 * y[t - 1] + rng.standard_normal(2) * 0.1

index = pd.date_range("2000-01-01", periods=T, freq="QS")
data = VARData(endog=y, endog_names=["gdp", "inflation"], index=index)
data

VARData(endog=array([[ 0.00000000e+00,  0.00000000e+00],
       [ 3.04717080e-02, -1.03998411e-01],
       [ 9.02809736e-02,  4.20572663e-02],
       [-1.49963032e-01, -1.09189318e-01],
       [-6.21974757e-02, -8.62189180e-02],
       [-3.27788536e-02, -1.28413852e-01],
       [ 7.15503707e-02,  1.35722677e-02],
       [ 4.23782551e-02,  1.19510255e-01],
       [ 6.79400618e-02, -2.61741190e-02],
       [ 7.08451093e-02, -1.08975320e-01],
       [ 1.23267585e-01, -5.94802509e-02],
       [ 4.31475560e-02, -9.78330799e-02],
       [ 1.43827912e-01, -6.43694882e-02],
       [ 2.90811737e-02, -6.73980991e-02],
       [ 6.77715054e-02,  2.84535687e-03],
       [ 7.51590139e-02,  4.45047787e-02],
       [ 2.51744267e-01, -1.83891123e-02],
       [ 7.46478606e-02, -9.05718290e-02],
       [ 9.89218726e-02,  6.76113148e-02],
       [ 3.80661905e-02, -5.02099903e-02],
       [-6.34150263e-02,  3.99542836e-02],
       [ 4.26179040e-02,  7.42925686e-02],
       [-4.52420187e-02,  6.03624166e-02

## Select lag order

Use information criteria to find the optimal lag length.

In [3]:
ic = select_lag_order(data, max_lags=8)
print(f"AIC: {ic.aic}, BIC: {ic.bic}, HQ: {ic.hq}")
ic.summary()

AIC: 1, BIC: 1, HQ: 1


,aic,bic,hq
lag,,,
1,-9.368291,-9.268995,-9.328103
2,-9.350139,-9.184065,-9.282918
3,-9.334984,-9.101660,-9.240533
4,-9.332458,-9.031407,-9.210578
5,-9.319690,-8.950429,-9.170180
6,-9.288078,-8.850118,-9.110736
7,-9.247516,-8.740362,-9.042135
8,-9.259885,-8.683037,-9.026257


## Specify and estimate the model

Create a VAR specification and fit it. We use a small number of draws here for speed.

In [4]:
spec = VAR(lags="bic", prior="minnesota")
sampler = NUTSSampler(draws=500, tune=500, chains=2, cores=1, random_seed=42)
fitted = spec.fit(data, sampler=sampler)
fitted

Initializing NUTS using jitter+adapt_diag...
Sequential sampling (2 chains in 1 job)
NUTS: [intercept, B, sigma_sd, tril_offdiag]


/Users/thomaspinder/Developer/litterman/.venv/lib/python3.11/site-packages/rich/live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Sampling 2 chains for 500 tune and 500 draw iterations (1_000 + 1_000 draws total) took 2 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics


FittedVAR(n_lags=1, n_vars=2, n_draws=500, n_chains=2)

## Inspect the posterior

Access the ArviZ InferenceData directly for diagnostics.

In [5]:
import arviz as az

az.summary(fitted.idata, var_names=["B", "intercept"])

,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
"B[0, 0]",0.606,0.059,0.504,0.722,0.002,0.002,963.0,711.0,1.0
"B[0, 1]",0.006,0.042,-0.067,0.084,0.001,0.001,1309.0,771.0,1.0
"B[1, 0]",0.003,0.040,-0.071,0.077,0.001,0.001,1457.0,754.0,1.0
"B[1, 1]",0.602,0.057,0.498,0.710,0.002,0.002,892.0,802.0,1.0
intercept[0],0.002,0.007,-0.011,0.016,0.000,0.000,1390.0,684.0,1.0
intercept[1],-0.002,0.006,-0.014,0.010,0.000,0.000,1115.0,788.0,1.0
